# Entropy Search: Edge-Triangle

This notebook inspects the edge-triangle target comparison and the coarse entropy grid. The grid is deliberately small because broad rectangular `(e,t)` grids contain many infeasible or numerically hard cells.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "graphon_space").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATA = ROOT / "outputs" / "final" / "data"
DATA

In [ ]:
compare = pd.read_parquet(DATA / "compare_triangle_e035_t002.parquet")
grid = pd.read_parquet(DATA / "entropy_grid_triangle.parquet")

compare_cols = ["family", "success", "entropy", "edge_residual", "t_residual", "symmetry_class"]
compare[compare_cols].sort_values(["success", "entropy"], ascending=[False, False])

In [ ]:
ok = compare[compare["success"]].copy()
winner = ok.loc[ok["entropy"].idxmax()]

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = np.where(compare["success"], "#2166ac", "#b2182b")
ax.bar(compare["family"], compare["entropy"], color=colors)
ax.set_ylabel("entropy")
ax.set_title("Target comparison: e=0.35, t=0.02")
ax.tick_params(axis="x", rotation=20)
display(winner[["family", "entropy", "symmetry_class", "edge_residual", "t_residual"]])
fig

In [ ]:
grid_ok = grid[grid["success"]].copy()
pd.Series({
    "grid records": len(grid),
    "successful records": len(grid_ok),
    "target cells with success": grid_ok.groupby(["target_e", "target_t"]).ngroups if len(grid_ok) else 0,
})

In [ ]:
if len(grid_ok):
    idx = grid_ok.groupby(["target_e", "target_t"])["entropy"].idxmax()
    winners = grid_ok.loc[idx].sort_values(["target_e", "target_t"])
    display(winners[["target_e", "target_t", "family", "entropy", "symmetry_class"]])

    fig, ax = plt.subplots(figsize=(6.5, 4.8))
    sc = ax.scatter(winners["target_e"], winners["target_t"], c=winners["entropy"], s=80, cmap="magma")
    ax.set_xlabel("edge density e")
    ax.set_ylabel("triangle density t")
    ax.set_title("Successful coarse entropy-grid cells")
    fig.colorbar(sc, ax=ax, label="best entropy")
    display(fig)
else:
    print("No successful grid cells in this coarse run.")